# Spark Structured Streaming
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/07_BigData_Spark/spark_structured_streaming.ipynb)

Structured Streaming treats a live data feed as an unbounded table: you write normal DataFrame queries and Spark runs them incrementally as new data arrives.

We simulate a live event stream with the built-in `rate` source and aggregate events in tumbling windows - everything runs locally in Colab.

In [ ]:
!pip install -q pyspark

## 1. Start a streaming DataFrame

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.master("local[*]")
         .appName("streaming-demo").getOrCreate())
spark.streams.resetTerminated()

events = (spark.readStream.format("rate")
          .option("rowsPerSecond", 10)
          .load())                       # columns: timestamp, value
print("isStreaming:", events.isStreaming)
events.printSchema()

## 2. Windowed aggregation -> memory sink

In [ ]:
agg = (events
       .withColumn("bucket", (F.col("value") % 5))          # fake device id
       .groupBy(F.window("timestamp", "10 seconds"), "bucket")
       .count())

query = (agg.writeStream
         .outputMode("complete")            # complete | append | update
         .format("memory")
         .queryName("window_counts")
         .start())
print("query active:", query.isActive)

## 3. Watch results arrive batch by batch

In [ ]:
import time
time.sleep(25)                               # let ~2 windows fill

for _ in range(3):
    spark.sql("SELECT * FROM window_counts ORDER BY window") \
         .show(8, truncate=False)
    print("---")
    time.sleep(10)

Each micro-batch covers one 10-second window. In production you would swap `rate` for Kafka/files/socket:

```python
spark.readStream.format("kafka") \
     .option("kafka.bootstrap.servers", "broker:9092") \
     .option("subscribe", "events").load()
```

## 4. Watermarks - handling late data

In [ ]:
watermarked = (events
    .withWatermark("timestamp", "30 seconds")            # grace period
    .groupBy(F.window("timestamp", "10 seconds"), F.col("value") % 3)
    .count())
q2 = watermarked.writeStream.outputMode("update") \
        .format("memory").queryName("wm_counts").start()
time.sleep(15)
spark.sql("SELECT * FROM wm_counts LIMIT 5").show()
for q in spark.streams.active:
    q.stop()                                             # cleanup
print("streams stopped:", not spark.streams.anyActive)

**Output modes**
| mode | what lands in the sink |
|---|---|
| append | only finalized rows (needs watermark for aggregations) |
| update | changed rows since last batch |
| complete | entire result table every batch |

Checkpointing (`checkpointLocation`) makes streams fault-tolerant - exactly-once semantics against idempotent sinks.